## 🧠🔗 SciKGExtract: Agentic Pipeline for Scientific Knowledge Graph Extraction - Tutorials!

### Tutorial 4: Knowledge Extraction with Normalization and Iterative Refinement from Scientific Texts
Welcome to the fourth tutorial of SciKGExtract! In this tutorial, we will demonstrate structured knowledge extraction with normalization and iterative refinement from scientific texts using the SciKGExtract framework. We will explore how to refine and update the extracted knowledge iteratively to enhance its accuracy and completeness. Additionally, we will show the use of feedback agent to formulate evaluation results from the reflection agent and instruct extraction agent for further refinement.

**Prerequisites:**
Before you begin, ensure you have completed the previous tutorials, especially:
- Tutorial 1: [SciKGExtract_Knowledge_Extraction](01_SciKGExtract_Knowledge_Extraction.ipynb)
- Tutorial 2: [SciKGExtract_Knowledge_Extraction_Normalization](02_SciKGExtract_Knowledge_Extraction_Normalization.ipynb)
- Tutorial 3: [SciKGExtract_Knowledge_Extraction_Normalization_Reflection](03_SciKGExtract_Knowledge_Extraction_Normalization_Reflection.ipynb)

**Focus of this Tutorial:**
- Knowledge Extraction with Normalization and Iterative Refinement
- Role of Feedback Agent in Evaluation and Refinement
- Comparison of Extraction Results Before and After Refinement

#### 📋 Overview
We will cover the following sections in this tutorial:
1. [**Initial Configuration**](#section-1-initial-configurations): Setting up the environment, loading necessary libraries, and defining the sample scientific text.
2. [**Process Definition**](#section-2-process-definition): Defining the scientific process for which we want to perform structured knowledge extraction.
3. [**Knowledge Extraction with Normalization without Refinement**](#section-3-knowledge-extraction-with-normalization-without-refinement): Performing knowledge extraction with normalization but without iterative refinement.
4. [**Knowledge Extraction with Normalization and Iterative Refinement**](#section-4-knowledge-extraction-with-normalization-and-iterative-refinement): Performing knowledge extraction with normalization and iterative refinement.
5. [**Evaluation of Extraction Results**](#section-5-evaluation-of-extraction-results): Evaluating and comparing the extraction results before and after refinement.
6. [**Next Steps**](#section-6-next-steps): Breif overview of next tutorials in the series.

#### ⚙️ Section 1: Initial Configurations <a id="section-1-initial-configurations"></a>
We will start with some initial configurations necessary for the extraction process. The configurations includes loading required libraries, setting up the LLM models and defining the input and output directories.

##### Import Necessary Packages and Modules

In [1]:
# Append the parent directory to sys.path
import sys
sys.path.append("../..")

# Apply nest_asyncio to allow nested event loops
import nest_asyncio
nest_asyncio.apply()

In [2]:
# Python Imports
import json

# SciKG-Extract Config Imports
from scikg_extract.config.process.processConfig import ProcessConfig
from scikg_extract.config.llm.envConfig import EnvConfig

# Import Utilities
from scikg_extract.utils.log_handler import LogHandler
from scikg_extract.utils.file_utils import read_json_file, read_text_file

# Import Orchestrator Agent
from scikg_extract.agents.orchestrator_agent import orchestrate_extraction_workflow

# Import Configurations
from scikg_extract.config.agents.orchestrator import OrchestratorConfig
from scikg_extract.config.agents.workflow import WorkflowConfig

# Import Data Models
from data.models.schema.ALD_experimental_schema import ALDProcessList

##### Configure Logging

In [3]:
# Setup and Initialize Module Logging
logger = LogHandler.setup_module_logging("scikg_extract")

##### Setting Up LLM Model

In [18]:
# LLM to be used for structured knowledge extraction
llm_name_extraction = "gpt-5.1"

# LLM to be used for Normalization especially for chemical entities disambiguation
llm_name_normalization = "gpt-5"

# LLM to be used for Reflection/Evaluation of the extracted knowledge
llm_name_reflection = "deepseek-r1"

# LLM to be used for Feedback formulation based on the evaluation results
llm_name_feedback = "gpt-5"

In [ ]:
# Set OpenAI API Key and Organization ID
EnvConfig.OPENAI_api_key = '<insert-your-openai-key>'
EnvConfig.OPENAI_organization_id = '<insert-your-openi-organization-id>'

##### Input and Output Directories

In [19]:
# Scientific document containing ZnO ALD experimental processes in markdown format
scientific_docs_dir = "../../data/research-papers/ALD/markdown/ZnO-IGZO-papers/experimental-usecase/ZnO/ZnEt2 - H2O/2 Lujala et al.md"
scientific_document = read_text_file(scientific_docs_dir)

# ALD Process Schema path for experimental processes
process_schema_path = "../../data/schemas/ALD-experimental/ALD-experimental-schema.json"
process_schema = read_json_file(process_schema_path)

# Domain-expert curated examples for ZnO ALD processes
examples_path = "../../data/examples/Atomic-layer-deposition/ZnO/example1.txt"
examples = read_text_file(examples_path)

# Manually curated PubChem synonym to CID mapping dictionary
pubchem_lookup_dict_path = "../../data/resources/PubChem-Synonym-CID.json"
synonym_to_cid_mapping = read_json_file(pubchem_lookup_dict_path)

# PubChem LMDB database created from PubChem CID data dump
lmdb_pubchem_path = "../../data/external/pubchem/pubchem_cid_lmdb"

#### 🧪 Section 2: Process Definition <a id="section-2-process-definition"></a>
In this section, we will define the scientific process for which we want to perform structured knowledge extraction. We will use the Atomic Layer Deposition (ALD) of Zinc Oxide (ZnO) as our example process. We have developed a detailed process schema for [ALD experimental process](../../data/schemas/ALD-experimental/ALD-experimental-schema.json) using our previous work: [**SCHEMA-MINER PRO**](https://github.com/sciknoworg/schema-miner), and will use it to guide the extraction process. Additionally, we will provide domain-expert curated examples to help the extraction agent understand the specific entities and relationships we are interested in.

In [20]:
# Process Name
ProcessConfig.Process_name = "Atomic Layer Deposition"

ProcessConfig.Process_description = """
Atomic layer deposition (ALD) is a surface-controlled thin film deposition technique that can enable ultimate control over the film thickness, uniformity on large-area substrates and conformality on 3D (nano)structures. Each ALD cycle consists at least two half-cycles (but can be more complex), containing a precursor dose step and a co-reactant exposure step, separated by purge or pump steps. Ideally the same amount of material is deposited in each cycle, due to the self-limiting nature of the reactions of the precursor and co-reactant with the surface groups on the substrate. By carrying out a certain number of ALD cycles, the targeted film thickness can be obtained.

In this extraction task, we are focusing on ZnO (Zinc Oxide) thin film deposition via ALD. A ZnO ALD (Zinc Oxide Atomic Layer Deposition) process deposits thin ZnO films through sequential, self-limiting surface reactions between a zinc precursor and an oxidant. The process typically consists of repeating ALD cycles, each containing a precursor pulse (e.g., diethylzinc (DEZ), Zn(acac)₂, or Zn(thd)₂), a purge step, an oxidant pulse (commonly H₂O, O₃, or O₂ plasma), followed by another purge. These reactions form a conformal zinc-oxygen layer per cycle with precise thickness control. The aim of a ZnO ALD process is to produce high-quality, uniform, conformal ZnO films with controlled thickness, crystallinity (amorphous or polycrystalline depending on temperature), and stoichiometry.
"""

#### 📄 Section 3: Knowledge Extraction with Normalization without Refinement <a id="section-3-knowledge-extraction-with-normalization-without-refinement"></a>
In this section, we will perform knowledge extraction with normalization but without iterative refinement. We will inspect the extracted knowledge and the LLM-as-a-judge evaluation results to understand the initial performance of the extraction process.

##### Import the Validation Rubrics
We will import the necessary validation rubrics that will be used by the LLM-as-a-Judge to evaluate the quality of the extracted knowledge. These rubrics are an extension of the [YESciEval framework](https://github.com/sciknoworg/YESciEval).

In [21]:
# Validation Rubrics Imports
from scikg_extract.evaluation.rubrics.informativeness import Completeness, Correctness

##### Orchestrator Agent Configuration
The orchestrator agent configuration will include additional parameters for evaluation. The additions include specifying the LLM model to be used as LLM-as-a-Judge, and defining a list of evaluation rubrics to be used for assessing the extracted knowledge.

In [ ]:
# Initialize orchestrator configuration
orchestrator_config = OrchestratorConfig(
    extraction_llm=llm_name_extraction,
    normalization_llm=llm_name_normalization,
    process_schema=process_schema,
    scientific_document=scientific_document,
    examples=examples,
    extraction_data_model=ALDProcessList,
    pubchem_lmdb_path=lmdb_pubchem_path,
    synonym_to_cid_mapping=synonym_to_cid_mapping,
    reflection_llm=llm_name_reflection,
    rubrics=[Completeness, Correctness]
)

##### Workflow Configuration
The workflow configuration now enables normalization and evaluation flags to activate the respective components in the extraction pipeline.

In [23]:
# Initialize the Workflow configuration
workflow_config = WorkflowConfig(normalize_extracted_data=True, clean_extracted_data=False, validate_extracted_data=True, refine_extracted_data=False)

##### Execute the Extraction Workflow with Normalization and Evaluation
With all the configurations set, we can now execute the knowledge extraction workflow that includes extraction, normalization, and evaluation. The orchestrator agent will oversee the entire process, ensuring that each component functions correctly and that the final output is a high-quality structured knowledge representation extracted from the scientific text and evaluation results.

In [24]:
# Extract knowledge using the orchestrator agent
final_state = orchestrate_extraction_workflow(orchestrator_config, workflow_config)

# Get the extracted knowledge from the final state
normalized_extracted_knowledge = final_state["normalized_json"]

2026-01-21 15:45:13 - scikg_extract.agents.orchestrator_agent - INFO - Starting Orchestrator Agent for extraction workflow...
2026-01-21 15:45:13 - scikg_extract.agents.orchestrator_agent - INFO - Compiled the orchestractor workflow graph.
2026-01-21 15:45:13 - scikg_extract.agents.orchestrator_agent - INFO - Invoking the orchestrator workflow...
2026-01-21 15:45:13 - scikg_extract.agents.extraction_agent - INFO - Starting knowledge extraction agent...
2026-01-21 15:45:13 - scikg_extract.agents.extraction_agent - INFO - Executing the extraction StateGraph for structured knowledge extraction...
2026-01-21 15:45:13 - scikg_extract.tools.extraction.structured_knowledge_extraction - INFO - Starting structured knowledge extraction tool...
2026-01-21 15:45:49 - scikg_extract.tools.extraction.json_validator - INFO - Starting JSON validation tool...
2026-01-21 15:45:49 - scikg_extract.tools.extraction.pubchem_normalization - INFO - Starting PubChem normalization tool...
2026-01-21 15:48:19 - s

##### Display Extracted Knowledge
Since the extracted normalized knowledge is based on the defined process schema which is very complex, nested and can contain multiple processes, we will display different parts of the extracted knowledge separately for better clarity and understanding. Specifically, we will display the ALD System, Reactants Selection and deposition temperature from process parameters of the first extracted process. The complete extracted and normalized knowledge can be accessed using `normalized_extracted_knowledge` variable.

In [26]:
print(f"Total Processes Extracted: {len(normalized_extracted_knowledge['processes'])}\n")

print("### ALD System ###\n")
ald_system = normalized_extracted_knowledge["processes"][0]["aldSystem"]
print(json.dumps(ald_system, separators=(',', ':')))

print("\n### Reactants Selection ###\n")
reactants_selection = normalized_extracted_knowledge["processes"][0]["reactantSelection"]
print(json.dumps(reactants_selection, separators=(',', ':')))

print("\n### Process Parameters - Deposition Temperature ###\n")
process_parameters_temperature = normalized_extracted_knowledge["processes"][0]["processParameters"]["temperature"]
print(json.dumps(process_parameters_temperature, separators=(',', ':')))

Total Processes Extracted: 1

### ALD System ###

{"aldMethod":[{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"method":"TALD"}],"materialDeposited":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]}}

### Reactants Selection ###

{"precursor":[{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"precursor":{"value":"dimethyl zinc (DMZn, (CH3)2Zn)","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/6093185"]}},{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"precursor":{"value":"diethyl zinc (DEZn, (C2H5)2Zn)","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/11185","https://pubchem.ncbi.nlm.nih.gov/compound/101667988"]}},{"compound":{"value":"Al","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/5359268"]},"precursor":{"value":"trimethyl aluminum (TMAl, (CH3)3Al)","sameAs":[]}}],"coReactant":[{"compound":{"value":"ZnO","sameAs":["htt

##### Display Evaluation Results
We will display now the evaluation results generated by the reflection agent using the specified rubrics. This evaluation is without any iterative refinement and will help us understand the initial quality of the extracted knowledge.

In [27]:
# Get the evaluation results from the final state
evaluation_results = final_state["evaluation_results"]

print("### Evaluation Results -- Completeness ###\n")
print(json.dumps(evaluation_results["completeness"], separators=(',', ':')))

print("\n### Evaluation Results -- Correctness ###\n")
print(json.dumps(evaluation_results["correctness"], separators=(',', ':')))

### Evaluation Results -- Completeness ###

{"rating":"2","rationale":"The extraction has several issues: 1) The article describes multiple processes (undoped ZnO and Al-doped ZnO at different temperatures and precursors), but only one process is extracted. 2) Required properties like 'flowRates' for precursors are missing (set to 0) despite the article mentioning carrier gas flow. 3) The 'supercycleDesign' incorrectly represents doping as a separate compound cycle rather than pulse replacement. 4) Many properties (e.g., optical band gap, roughness) are set to 0 without article support. 5) The 'nucleationPeriod' is set to 0 without justification. 6) The 'saturation' flag is set to false without evidence. These errors and omissions violate schema constraints and reduce traceability."}

### Evaluation Results -- Correctness ###

{"rating":"2","rationale":"The extraction has several significant issues: 1. Incorrect pressure unit: The article states pressure as 0.5-0.8 kPa, but extraction 

#### 📄 Section 4: Knowledge Extraction with Normalization and Iterative Refinement <a id="section-4-knowledge-extraction-with-normalization-and-iterative-refinement"></a>
In this section, we will perform knowledge extraction with normalization and iterative refinement. We will inspect the extracted knowledge and the LLM-as-a-judge evaluation results to understand the performance of the extraction process after refinement. 

##### Orchestrator Agent Configuration
The orchestrator agent configuration will include additional parameters for evaluation. The additions include specifying the LLM model to be used as LLM-as-a-Judge, and defining a list of evaluation rubrics to be used for assessing the extracted knowledge.

In [ ]:
# Initialize orchestrator configuration
orchestrator_config = OrchestratorConfig(
    extraction_llm=llm_name_extraction,
    normalization_llm=llm_name_normalization,
    process_schema=process_schema,
    scientific_document=scientific_document,
    examples=examples,
    extraction_data_model=ALDProcessList,
    pubchem_lmdb_path=lmdb_pubchem_path,
    synonym_to_cid_mapping=synonym_to_cid_mapping,
    reflection_llm=llm_name_reflection,
    rubrics=[Completeness, Correctness],
    feedback_llm=llm_name_feedback
)

##### Workflow Configuration
The workflow configuration now enables normalization and evaluation flags to activate the respective components in the extraction pipeline.

In [29]:
# Initialize the Workflow configuration
workflow_config = WorkflowConfig(normalize_extracted_data=True, clean_extracted_data=False, validate_extracted_data=True, refine_extracted_data=True, total_validation_retries=3)

##### Execute the Extraction Workflow with Normalization and Evaluation
With all the configurations set, we can now execute the knowledge extraction workflow that includes extraction, normalization, and evaluation. The orchestrator agent will oversee the entire process, ensuring that each component functions correctly and that the final output is a high-quality structured knowledge representation extracted from the scientific text and evaluation results.

In [30]:
# Extract knowledge using the orchestrator agent
final_state = orchestrate_extraction_workflow(orchestrator_config, workflow_config)

# Get the extracted knowledge from the final state
normalized_extracted_knowledge = final_state["normalized_json"]

2026-01-21 15:49:28 - scikg_extract.agents.orchestrator_agent - INFO - Starting Orchestrator Agent for extraction workflow...
2026-01-21 15:49:28 - scikg_extract.agents.orchestrator_agent - INFO - Compiled the orchestractor workflow graph.
2026-01-21 15:49:28 - scikg_extract.agents.orchestrator_agent - INFO - Invoking the orchestrator workflow...
2026-01-21 15:49:28 - scikg_extract.agents.extraction_agent - INFO - Starting knowledge extraction agent...
2026-01-21 15:49:28 - scikg_extract.agents.extraction_agent - INFO - Executing the extraction StateGraph for structured knowledge extraction...
2026-01-21 15:49:28 - scikg_extract.tools.extraction.structured_knowledge_extraction - INFO - Starting structured knowledge extraction tool...
2026-01-21 15:50:02 - scikg_extract.tools.extraction.json_validator - INFO - Starting JSON validation tool...
2026-01-21 15:50:02 - scikg_extract.tools.extraction.pubchem_normalization - INFO - Starting PubChem normalization tool...
2026-01-21 15:51:33 - s

##### Display Extracted Knowledge
Since the extracted normalized knowledge is based on the defined process schema which is very complex, nested and can contain multiple processes, we will display different parts of the extracted knowledge separately for better clarity and understanding. Specifically, we will display the ALD System, Reactants Selection and deposition temperature from process parameters of the first extracted process. The complete extracted and normalized knowledge can be accessed using `normalized_extracted_knowledge` variable.

In [ ]:
print(f"Total Processes Extracted: {len(normalized_extracted_knowledge['processes'])}\n")

print("### ALD System ###\n")
ald_system = normalized_extracted_knowledge["processes"][0]["aldSystem"]
print(json.dumps(ald_system, separators=(',', ':')))

print("\n### Reactants Selection ###\n")
reactants_selection = normalized_extracted_knowledge["processes"][0]["reactantSelection"]
print(json.dumps(reactants_selection, separators=(',', ':')))

print("\n### Process Parameters - Deposition Temperature ###\n")
process_parameters_temperature = normalized_extracted_knowledge["processes"][0]["processParameters"]["temperature"]
print(json.dumps(process_parameters_temperature, separators=(',', ':')))

### ALD System ###

{"aldMethod":[{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"method":"TALD"}],"materialDeposited":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]}}

### Reactants Selection ###

{"precursor":[{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"precursor":{"value":"DMZn","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/6093185"]}},{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"precursor":{"value":"DEZn","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/11185","https://pubchem.ncbi.nlm.nih.gov/compound/101667988"]}}],"coReactant":[{"compound":{"value":"ZnO","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/14806"]},"coReactant":{"value":"H2O","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/962"]}}],"carrierGas":{"value":"N2","sameAs":["https://pubchem.ncbi.nlm.nih.gov/compound/947"]},"purgingGas":{"value

##### Display Evaluation Results
We will display now the evaluation results generated by the reflection agent using the specified rubrics. This evaluation is without any iterative refinement and will help us understand the initial quality of the extracted knowledge.

In [17]:
# Get the evaluation results from the final state
evaluation_results = final_state["evaluation_results"]

print("### Evaluation Results -- Completeness ###\n")
print(json.dumps(evaluation_results["completeness"], separators=(',', ':')))

print("\n### Evaluation Results -- Correctness ###\n")
print(json.dumps(evaluation_results["correctness"], separators=(',', ':')))

### Evaluation Results -- Completeness ###

{"rating":"2","rationale":"The extraction has several issues. First, the article describes both undoped and Al-doped ZnO processes, but the extraction only includes one process for ZnO without specifying doping. The Al doping is a key aspect, with TMAl as the precursor, but it's missing in the extracted data. Second, the growth rate varies with temperature and precursor (0.5-2.5 A/cycle), but the extraction only lists 0.5 A/cycle without context. Third, the substrate temperature range is 120-350\u00b0C, but the extraction only lists 250\u00b0C. Fourth, the article mentions multiple precursors (DMZn, DEZn) and co-reactant (H2O), but the extraction lists both precursors under one compound without distinction. Fifth, the electrical properties include resistivity for doped films (e.g., 8e-4 ohm.cm), but the extraction lists 0.001 ohm.cm without specifying doping. Sixth, the film thickness of 750 nm is for a doped film, but the extraction doesn't 